# Step 11 — Final Results Aggregation

**Input:**
- `data/output/10_building_level_redistributed.gpkg`
- `data/input/zone_targets.gpkg`

**Output:** `data/output/11_final_results.gpkg`

Aggregates building-level assigned activity counts back to zone level,
merges with official zone targets for comparison, and exports the final dataset.

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path('..').resolve()))
from config import *

import geopandas as gpd
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
print('Config loaded')

## 1. Load redistribution results and zones

In [ ]:
buildings = gpd.read_file(REDISTRIBUTION_FILE)
zones     = gpd.read_file(ZONE_TARGETS_FILE, layer=ZONE_LAYER if ZONE_LAYER else None).to_crs(TARGET_CRS)
zones = zones.reset_index(drop=True)
zones['zone_id'] = zones.index

print(f'Buildings: {len(buildings):,}')
print(f'Zones:     {len(zones):,}')

## 2. Spatial join buildings to zones

In [ ]:
bld_cents = buildings.copy()
bld_cents['geometry'] = buildings.geometry.centroid

bld_with_zone = gpd.sjoin(bld_cents, zones[['zone_id', 'geometry']], how='left', predicate='within')
bld_with_zone = bld_with_zone.drop_duplicates(subset='gml_id')
print(f'Buildings matched to a zone: {bld_with_zone["zone_id"].notna().sum():,}')

## 3. Aggregate to zone level

In [ ]:
assigned_cols = [f'assigned_{act.replace("-", "_")}' for act in ZONE_ACTIVITY_COLUMNS
                 if f'assigned_{act.replace("-", "_")}' in bld_with_zone.columns]

zone_agg = (
    bld_with_zone[bld_with_zone['zone_id'].notna()]
    .groupby('zone_id')[assigned_cols]
    .sum()
    .reset_index()
)

# Merge with zones
zones_result = zones.merge(zone_agg, on='zone_id', how='left')
zones_result[assigned_cols] = zones_result[assigned_cols].fillna(0)

print('Zone-level aggregated results:')
print(zones_result[assigned_cols].describe().round(1))

## 4. Save final results

In [ ]:
buildings.to_file(FINAL_RESULTS_FILE, driver='GPKG', layer='buildings')
zones_result.to_file(FINAL_RESULTS_FILE, driver='GPKG', layer='zones')

print(f'Saved building-level results and zone-level aggregation → {FINAL_RESULTS_FILE}')
print('\nPipeline complete!')